# Klinik Karar Eğrisi Analizi (Decision Curve Analysis - DCA)
**Chest X-Ray Tiered Classification · Klinik Fayda Değerlendirmesi**

Bu defter, kademeli chest X-ray pneumothorax sınıflandırma modelimizin klinik kararlardaki faydasını **Net Benefit (Net Fayda)** metriği üzerinden değerlendirmektedir.

### Net Benefit Formülasyonu:
$$\text{Net Benefit} = \frac{\text{True Positives}}{N} - \frac{\text{False Positives}}{N} \times \left(\frac{p_t}{1 - p_t}\right)$$

Burada $p_t$ klinik karar eşiğidir (threshold probability). Karar eğrisi analizi, modelin "herkese müdahale et" (Treat All) veya "hiç kimseye müdahale etme" (Treat None) stratejilerine kıyasla sağladığı ek net faydayı ortaya koyar.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# REAL predictions for Tier 1, the deep Tier 2 backbone, and the routed tiered system.
_PRED = next((p for p in ["../outputs/results/tiered_predictions.csv",
                          "outputs/results/tiered_predictions.csv"]
              if os.path.exists(p)), None)
if _PRED is None:
    raise FileNotFoundError(
        "tiered_predictions.csv not found. Generate REAL predictions first:\n"
        "    python scripts/export_predictions.py --tier2-backbone ark_plus\n"
        "(run on Colab after the models are trained/restored)."
    )
df = pd.read_csv(_PRED)
y_true = df["y_true"].to_numpy()
n_samples = len(df)
probs_t1 = df["tier1_prob"].to_numpy()
probs_t2 = df["tier2_prob"].to_numpy()
probs_tiered = df["tiered_prob"].to_numpy()
print(f"Loaded {n_samples} REAL predictions from {_PRED}.")


### DCA Hesaplama Fonksiyonu

In [ ]:
def calculate_net_benefit(y_true, y_probs, threshold):
    y_pred = (y_probs >= threshold).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    n = len(y_true)
    if threshold == 1.0:
        return 0.0
    return (tp / n) - (fp / n) * (threshold / (1 - threshold))

thresholds = np.linspace(0.05, 0.85, 100)
nb_treat_all = [calculate_net_benefit(y_true, np.ones(n_samples), t) for t in thresholds]
nb_treat_none = [0.0] * len(thresholds)
nb_t1 = [calculate_net_benefit(y_true, probs_t1, t) for t in thresholds]
nb_t2 = [calculate_net_benefit(y_true, probs_t2, t) for t in thresholds]
nb_tiered = [calculate_net_benefit(y_true, probs_tiered, t) for t in thresholds]


### Klinik Karar Eğrisinin Görselleştirilmesi
Aşağıdaki grafik, kademeli sistemimizin klinik karar mekanizmasındaki ek Net Benefit başarısını göstermektedir.

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(thresholds, nb_treat_all, '--', label='Treat All', color='red', alpha=0.6)
plt.plot(thresholds, nb_treat_none, '-', label='Treat None', color='black', alpha=0.8)
plt.plot(thresholds, nb_t1, label='Tier 1 only (MobileNetV2)', color='orange', alpha=0.8)
plt.plot(thresholds, nb_t2, label='Tier 2 deep backbone', color='blue', linewidth=2)
plt.plot(thresholds, nb_tiered, label='Tiered System (Routed)', color='green', linewidth=2.5, linestyle='-.')
plt.xlim(0.05, 0.85)
plt.ylim(-0.05, 0.40)
plt.xlabel('Threshold Probability (Pt)')
plt.ylabel('Net Benefit')
plt.title('Clinical Decision Curve Analysis (DCA) - real test predictions')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
